In [8]:
# Import necessary libraries
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
from scipy import signal
import os

In [9]:
def remove_noise_comprehensive(input_file, output_file):
    """
    Enhanced noise removal with more aggressive parameters
    """
    print(f"Loading audio file: {input_file}")
    audio, sr = librosa.load(input_file, sr=None)
    
    print(f"Sample rate: {sr} Hz")
    print(f"Duration: {len(audio) / sr:.2f} seconds")
    
    # Step 1: Initial aggressive spectral noise reduction
    print("Step 1: Aggressive spectral noise reduction...")
    audio = nr.reduce_noise(
        y=audio, 
        sr=sr, 
        stationary=True,
        prop_decrease=1.0,  # Maximum reduction
        freq_mask_smooth_hz=500,  # Smooth frequency masking
        time_mask_smooth_ms=50    # Smooth time masking
    )
    
    # Step 2: High-pass filter to remove low frequency rumble
    print("Step 2: Removing low-frequency rumble...")
    nyquist = sr / 2
    b, a = signal.butter(5, 100 / nyquist, btype='high')  # Remove everything below 100Hz
    audio = signal.filtfilt(b, a, audio)
    
    # Step 3: Bandpass Filter (100Hz - 7000Hz for cleaner voice)
    print("Step 3: Applying bandpass filter...")
    low = 100 / nyquist
    high = 7000 / nyquist
    b, a = signal.butter(5, [low, high], btype='band')
    audio = signal.filtfilt(b, a, audio)
    
    # Step 4: Multi-band noise reduction
    print("Step 4: Multi-band noise reduction...")
    audio = nr.reduce_noise(
        y=audio, 
        sr=sr, 
        stationary=False,
        prop_decrease=0.9,
        n_fft=2048
    )
    
    # Step 5: Final pass with adaptive noise reduction
    print("Step 5: Final adaptive noise cleaning...")
    audio = nr.reduce_noise(
        y=audio, 
        sr=sr,
        stationary=False,
        prop_decrease=0.7,
        freq_mask_smooth_hz=200
    )
    
    # Normalize audio
    audio = audio / np.max(np.abs(audio)) * 0.95
    
    # Save the cleaned audio
    sf.write(output_file, audio, sr)
    print(f"\n✓ Cleaned audio saved to: {output_file}")
    
    return audio, sr

In [10]:
input_filename = 'Julias the story.wav'  # Your audio file
output_filename = 'cleaned_audio.wav'

print("🎵 Starting enhanced noise removal...\n")
cleaned_audio, sample_rate = remove_noise_comprehensive(input_filename, output_filename)
print(f"\n✓ Done! File size: {os.path.getsize(output_filename) / 1024:.2f} KB")

🎵 Starting enhanced noise removal...

Loading audio file: Julias the story.wav
Sample rate: 48000 Hz
Duration: 264.16 seconds
Step 1: Aggressive spectral noise reduction...
Step 2: Removing low-frequency rumble...
Step 3: Applying bandpass filter...
Step 4: Multi-band noise reduction...
Step 5: Final adaptive noise cleaning...

✓ Cleaned audio saved to: cleaned_audio.wav

✓ Done! File size: 24764.69 KB
